# Motion Feature Extraction - Single Record Test

This notebook demonstrates how to extract motion features from specific records for testing and debugging.

In [1]:
import sys
sys.path.insert(0, '/tf/01_code/mylittlecodes/SleepVST_baseline')

from src.data.preprocess.motion_test_helper import (
    extract_motion_features_from_record,
    get_video_info,
    compare_motion_features,
    batch_extract_records
)
import numpy as np
import matplotlib.pyplot as plt

## 1. Get Video Information

First, let's check the basic information about a video.

In [3]:
# Specify the record ID you want to test
record_id = 'A2020-EM-01-0184'

# Get video info
video_info = get_video_info(record_id)
print(f"Video info for {record_id}:")
for key, value in video_info.items():
    print(f"  {key}: {value}")

Video info for A2020-EM-01-0184:
  record_id: A2020-EM-01-0184
  path: /tf/00_data/#_2021_Sleep_Video/A2020-EM-01-0184/A2020-EM-01-0184_video_01.mp4
  fps: 4.930555555555555
  frame_count: 141853
  width: 640
  height: 480
  duration_seconds: 28770.18591549296
  duration_minutes: 479.5030985915493
  expected_epochs: 959.0061971830986
  expected_motion_steps: 141852


## 2. Extract Motion Features

Extract motion features from the specified record.

In [4]:
# Extract motion features
features = extract_motion_features_from_record(
    record_id=record_id,
    skip_existing=False,  # Set to True to skip if already exists
    verbose=True
)

if features:
    print(f"\nExtracted features:")
    print(f"  Number of features: {len(features)}")
    print(f"  Feature names: {list(features.keys())}")
    print(f"  Feature shape (example): {next(iter(features.values())).shape}")

2025-10-23 12:28:21,943 - INFO - Extracting motion features for record: A2020-EM-01-0184
2025-10-23 12:28:21,988 - INFO - Processing A2020-EM-01-0184_video_01.mp4 with 141852 steps at target FPS 4


KeyboardInterrupt: 

## 3. Compare with Expected Values

Compare the extracted features with expected values based on video properties.

In [ ]:
# Compare features
comparison = compare_motion_features(record_id, features)

if comparison:
    print(f"\nDetailed comparison results:")
    print(f"  Expected motion steps: {comparison['expected_motion_steps']}")
    print(f"  Actual feature length: {comparison['feature_length']}")
    print(f"  Difference: {comparison['difference']}")
    print(f"  Feature length to epochs ratio: {comparison['feature_length_to_epochs']:.6f}")

## 4. Visualize Motion Features

Visualize the extracted motion features over time.

In [ ]:
if features:
    # Select some features to visualize
    feature_names = ['f1@30s_Head', 'f1@30s_Body', 'f1@30s_Outer']
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 10))
    fig.suptitle(f'Motion Features for {record_id}', fontsize=16)
    
    for i, feat_name in enumerate(feature_names):
        if feat_name in features:
            axes[i].plot(features[feat_name])
            axes[i].set_title(feat_name)
            axes[i].set_xlabel('Time Steps (4 fps)')
            axes[i].set_ylabel('Feature Value')
            axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nFeature statistics:")
    for feat_name in feature_names:
        if feat_name in features:
            feat_data = features[feat_name]
            print(f"\n{feat_name}:")
            print(f"  Min: {np.min(feat_data):.4f}")
            print(f"  Max: {np.max(feat_data):.4f}")
            print(f"  Mean: {np.mean(feat_data):.4f}")
            print(f"  Std: {np.std(feat_data):.4f}")

## 5. Batch Process Multiple Records

Process multiple records at once.

In [ ]:
# List of records to process
test_record_ids = ['A-0038', 'A-0039', 'A-0040']

# Batch extract
results = batch_extract_records(
    record_ids=test_record_ids,
    skip_existing=True  # Skip already processed records
)

# Print results
print("\nResults:")
for record_id, result in results.items():
    if result['success']:
        print(f"  {record_id}: Success - {result['feature_count']} features, length {result['feature_length']}")
    else:
        print(f"  {record_id}: Failed - {result['error']}")

## 6. Compare Multiple Records

Compare motion feature characteristics across multiple records.

In [ ]:
# Compare feature lengths across records
if results:
    record_ids = [rid for rid, r in results.items() if r['success']]
    feature_lengths = [r['feature_length'] for r in results.values() if r['success']]
    
    plt.figure(figsize=(10, 6))
    plt.bar(record_ids, feature_lengths)
    plt.xlabel('Record ID')
    plt.ylabel('Feature Length')
    plt.title('Motion Feature Length Comparison')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Print comparison table
    print(f"\n{'Record ID':<15} {'Feature Length':<15} {'Expected Epochs':<15}")
    print("="*45)
    for record_id in record_ids:
        video_info = get_video_info(record_id)
        if video_info:
            feat_len = results[record_id]['feature_length']
            expected_epochs = video_info['expected_epochs']
            print(f"{record_id:<15} {feat_len:<15} {expected_epochs:<15.2f}")

## 7. Analyze Specific Feature Patterns

Analyze specific motion feature patterns for a single record.

In [ ]:
# Load features if not already loaded
if features is None:
    from pathlib import Path
    output_dir = "/tf/01_code/mylittlecodes/SleepVST_baseline/data/motionfeatures_test"
    output_file = Path(output_dir) / f"{record_id}_motion_features.npy"
    if output_file.exists():
        features = np.load(output_file, allow_pickle=True).item()

if features:
    # Analyze all time-based features
    time_features = {k: v for k, v in features.items() if 'f1@' in k or 'f2@' in k}
    
    fig, axes = plt.subplots(len(time_features), 1, figsize=(15, 4*len(time_features)))
    if len(time_features) == 1:
        axes = [axes]
    
    fig.suptitle(f'All Time-Based Features for {record_id}', fontsize=16)
    
    for i, (feat_name, feat_data) in enumerate(time_features.items()):
        axes[i].plot(feat_data, linewidth=0.5)
        axes[i].set_title(feat_name)
        axes[i].set_xlabel('Time Steps (4 fps)')
        axes[i].set_ylabel('Feature Value')
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()